# Fase 3+4 — Validación, ONNX export y empaquetado

Notebook corto (~15 min) para correr **después** de que `03_kaggle_pipeline.ipynb` ya entrenó.
No reentrena nada — solo carga `best.pt`, evalúa, exporta a ONNX y empaqueta.

## Antes de correr

1. Extraé `_output_.zip` que bajaste del run anterior.
2. Dentro vas a encontrar `runs/barbell_detector/weights/best.pt`.
3. Subí ese `best.pt` a Kaggle como nuevo Dataset (New Dataset > Upload file).
4. En este notebook: **+ Add Input** → buscá tu dataset → montará en `/kaggle/input/<nombre>/best.pt`.
5. Ajustá `BEST_PT_PATH` abajo con la ruta exacta.
6. **Save & Run All (Commit)** — GPU no es necesario para ONNX export pero acelera la validación.

In [ ]:
BEST_PT_PATH = "/kaggle/input/barbell-best-pt/best.pt"  # <- ajustá al nombre de tu dataset

In [ ]:
!pip install -q ultralytics roboflow
from kaggle_secrets import UserSecretsClient
ROBOFLOW_API_KEY = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")

In [ ]:
# Re-descarga solo los datasets públicos para tener data.yaml e imágenes de test.
# Mucho más rápido que el run original: no hay Autodistill, solo descargas de Roboflow (~5 min).
import os, glob, shutil, yaml, random
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
RAW_DIR = "/kaggle/working/raw_datasets"
os.makedirs(RAW_DIR, exist_ok=True)

SOURCES = [
    ("computer-vision-1bdzc", "barbell-detection-yolov5", 6, "barbell_a"),
    ("nkust-airs-lps", "nkust_barbell", 3, "barbell_b"),
    ("ecnu", "gym_001", 1, "gym_001"),
    ("gymitemidentifier", "gym_greyscale", 1, "gym_greyscale"),
]

downloaded = []
for workspace, project, version, folder in SOURCES:
    try:
        ds = rf.workspace(workspace).project(project).version(version).download(
            "yolov8", location=os.path.join(RAW_DIR, folder)
        )
        downloaded.append(folder)
        print(f"OK: {project}")
    except Exception as e:
        print(f"FALLÓ {project}: {e}")

In [ ]:
# Reconstruye data.yaml con las 9 clases finales y junta imágenes de test de los datasets públicos.
# Las 4 clases de Autodistill no tienen imágenes de test acá (eso es aceptable para esta validación rápida).
FINAL_CLASSES = ["barbell", "dumbbell", "kettlebell", "plate", "cable",
                 "barra_olimpica", "barra_hexagonal", "barra_curl", "maquina_polea"]
CLASS_ALIASES = {"barbell": "barbell", "Barbell": "barbell"}
MASTER_CLASSES = FINAL_CLASSES[:5]

TEST_DIR = "/kaggle/working/test_imgs"
os.makedirs(f"{TEST_DIR}/images", exist_ok=True)
os.makedirs(f"{TEST_DIR}/labels", exist_ok=True)

for folder in downloaded:
    data_yaml = yaml.safe_load(open(os.path.join(RAW_DIR, folder, "data.yaml")))
    src_classes = data_yaml["names"]
    id_map = {}
    for i, name in enumerate(src_classes):
        canon = CLASS_ALIASES.get(name, name)
        if canon in MASTER_CLASSES:
            id_map[i] = MASTER_CLASSES.index(canon)
    test_img_dir = os.path.join(RAW_DIR, folder, "test", "images")
    test_lbl_dir = os.path.join(RAW_DIR, folder, "test", "labels")
    if not os.path.isdir(test_img_dir):
        continue
    for img_path in glob.glob(os.path.join(test_img_dir, "*")):
        stem = os.path.splitext(os.path.basename(img_path))[0]
        lbl_path = os.path.join(test_lbl_dir, stem + ".txt")
        new_lines = []
        if os.path.exists(lbl_path):
            for line in open(lbl_path):
                parts = line.split()
                if parts and int(parts[0]) in id_map:
                    new_lines.append(f"{id_map[int(parts[0])]} {' '.join(parts[1:])}")
        if not new_lines:
            continue
        tag = os.path.basename(folder)
        shutil.copy(img_path, f"{TEST_DIR}/images/{tag}_{os.path.basename(img_path)}")
        open(f"{TEST_DIR}/labels/{tag}_{stem}.txt", "w").write("\n".join(new_lines))

data_yaml_path = "/kaggle/working/data.yaml"
yaml.safe_dump(
    {"train": TEST_DIR + "/images", "val": TEST_DIR + "/images", "test": TEST_DIR + "/images",
     "nc": len(FINAL_CLASSES), "names": FINAL_CLASSES},
    open(data_yaml_path, "w"),
)
print(f"data.yaml listo. Imágenes de test disponibles: {len(glob.glob(TEST_DIR + '/images/*'))}")

## Fase 3 — Validación con métricas reales

In [ ]:
from ultralytics import YOLO

best_model = YOLO(BEST_PT_PATH)
metrics = best_model.val(data=data_yaml_path, split="val")

print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print("\nPor clase (mAP50):")
for i, name in metrics.names.items():
    if i < len(metrics.box.ap50):
        print(f"  {name}: {metrics.box.ap50[i]:.4f}")
    else:
        print(f"  {name}: sin instancias en validación")

In [ ]:
# Ejemplos de detección sobre el set de test
from PIL import Image

test_images = glob.glob(os.path.join(TEST_DIR, "images", "*"))
sample = random.sample(test_images, k=min(10, len(test_images)))
pred_results = best_model.predict(sample, conf=0.25, save=True,
                                   project="/kaggle/working/runs", name="test_preview")
for r in pred_results:
    print(os.path.basename(r.path), "->", len(r.boxes), "detecciones")
    display(Image.fromarray(r.plot()[:, :, ::-1]).resize((480, 360)))

## Fase 4 — Export a ONNX (para medir viabilidad en navegador)

In [ ]:
onnx_path = best_model.export(format="onnx", imgsz=640)
size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
print(f"ONNX exportado: {onnx_path} — {size_mb:.1f} MB")

## Empaquetar — aparece en el tab Output del commit

In [ ]:
OUT_DIR = "/kaggle/working/resultado_entrenamiento"
os.makedirs(OUT_DIR, exist_ok=True)
shutil.copy(BEST_PT_PATH, OUT_DIR)
shutil.copy(onnx_path, OUT_DIR)
if os.path.isdir("/kaggle/working/runs/test_preview"):
    shutil.copytree("/kaggle/working/runs/test_preview",
                    os.path.join(OUT_DIR, "test_preview"), dirs_exist_ok=True)

with open(os.path.join(OUT_DIR, "resumen.txt"), "w") as f:
    f.write(f"mAP50: {metrics.box.map50:.4f}\n")
    f.write(f"mAP50-95: {metrics.box.map:.4f}\n")
    f.write(f"Tamaño ONNX: {size_mb:.1f} MB\n")
    f.write("\nPor clase:\n")
    for i, name in metrics.names.items():
        if i < len(metrics.box.ap50):
            f.write(f"  {name}: {metrics.box.ap50[i]:.4f}\n")
        else:
            f.write(f"  {name}: sin instancias en validación\n")

shutil.make_archive("/kaggle/working/resultado_entrenamiento", "zip", OUT_DIR)
print("Listo: resultado_entrenamiento.zip en el tab Output")